# Урок 6 — Честная проверка модели (Train / Test)

На прошлом уроке наш KNN показал точность 0.87. Но мы проверяли его на тех же пассажирах, которых он запомнил — это как **списать с ответов на контрольной**.

Сегодня научимся проверять модель **честно**.

Запускай ячейки по порядку: ▶ или **Shift+Enter**.

---
### Идея урока
- **Train (80%)** — на этих данных модель учится.
- **Test (20%)** — эти данные модель НЕ видит при обучении. Это «новая контрольная».
- Обучаем на train → проверяем на test → точность честная.

## Шаг 1. Готовим данные (как на прошлом уроке)

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df = df[["Pclass", "Sex", "Age", "Fare", "Survived"]].dropna()
df["Sex"] = (df["Sex"] == "male").astype(int)

X = df[["Pclass", "Sex", "Age", "Fare"]]
y = df["Survived"]
print("Данные готовы. Всего строк:", len(df))

## Шаг 2. Делим на train и test ✂️
`train_test_split` случайно разбивает данные на две части.
- `test_size=0.2` → 20% уходит в тест.
- `random_state=42` → фиксируем случайность, чтобы у всех было одинаково.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% данных — в тест
    random_state=42     # фиксируем случайное разбиение
)

print("Обучающих строк (train):", len(X_train))
print("Тестовых строк (test):", len(X_test))

## Шаг 3. Масштабируем
Важно: scaler «учим» (`fit`) ТОЛЬКО на train. Тест он не должен видеть заранее —
иначе это снова подглядывание в ответы.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # учим scaler на train и применяем
X_test = scaler.transform(X_test)         # к тесту — только применяем

print("Признаки выровнены.")

## Шаг 4. Обучаем модель на TRAIN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)   # учим ТОЛЬКО на обучающих данных

print("Модель обучена на train.")

## Шаг 5. Сравниваем: train vs test 👀
Главный момент урока. Проверим точность на обеих частях.

In [ ]:
from sklearn.metrics import accuracy_score

# точность на train (которые модель видела)
train_acc = accuracy_score(y_train, model.predict(X_train))

# точность на test (которые модель НЕ видела)
test_acc = accuracy_score(y_test, model.predict(X_test))

print("Точность на TRAIN:", round(train_acc, 2), "← модель эти данные видела")
print("Точность на TEST: ", round(test_acc, 2), "← честная оценка!")

**Что мы видим:** на train точность выше, чем на test. Это нормально!
Модель эти данные «помнит». А test показывает, как она справится с **новыми** людьми.

⚠️ Если train = 0.99, а test = 0.65 — модель не поняла, а **зазубрила**.

## ⭐ Для тех, кто хочет больше — честный лучший K
Теперь выбираем K по ТЕСТУ, а не по train. Бери нечётные K!

In [ ]:
for k in [1, 3, 5, 7, 9, 11]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"K={k}: точность на тесте {round(acc, 2)}")

## ✍️ Твоё задание
1. Найди K, который даёт лучшую точность **на тесте** (не на train!).
2. Запиши: какая точность на train и на test при K=5? Что больше? Почему?
3. Подумай: если бы данных было всего 10 строк — можно ли честно поделить на train/test?

---
**Вывод урока:** проверять модель на тех же данных, на которых она училась — нечестно,
это «списывание с ответов». Правильно: учим на **train**, проверяем на **test**.

Близкие train и test = модель поняла. Большой разрыв = зазубрила.
Это — главное правило всего ML.

**Домашка:** найди лучший K по тесту, сравни train/test при K=5, подумай про маленькие данные.